In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# INPUT AND OUTPUT PATHS
# ============================================================

INPUT_PATH = Path(
    "/kaggle/input/notebooks/notjustauser/"
    "01-smard-cleaning/smard_hourly_clean.parquet"
)

FEATURE_OUTPUT_PATH = Path(
    "/kaggle/working/smard_features.parquet"
)

FEATURE_CONFIG_PATH = Path(
    "/kaggle/working/feature_engineering_config.json"
)


# ============================================================
# FORECASTING CONFIGURATION
# ============================================================

CANONICAL_TIMEZONE = "UTC"
CALENDAR_TIMEZONE = "Europe/Berlin"

FORECAST_ORIGIN_HOUR_UTC = 12
LOOKBACK_HOURS = 168
FORECAST_HORIZON_HOURS = 24

PRICE_LAGS = [1, 24, 48, 168]
ROLLING_WINDOWS = [24, 168]

MAX_FEATURE_HISTORY_HOURS = max(
    max(PRICE_LAGS),
    max(ROLLING_WINDOWS),
)

MINIMUM_CLEAN_HISTORY_HOURS = (
    LOOKBACK_HOURS
    + MAX_FEATURE_HISTORY_HOURS
)

EXPECTED_NOTEBOOK1_COLUMNS = [
    "price_eur_mwh",
    "load_actual_mwh",
    "load_forecast_mwh",
]

EXPECTED_HISTORICAL_FORECAST_NANS = 97


# Keep the Notebook 01 output separate from the feature table.
clean_data = pd.read_parquet(INPUT_PATH)

In [2]:
def validate_notebook1_input(
    frame,
    expected_forecast_nans=None,
):
    """
    Validate that the loaded dataset follows the Notebook 01
    output contract.
    """
    errors = []

    # ------------------------------------------------------------
    # Schema
    # ------------------------------------------------------------

    if list(frame.columns) != EXPECTED_NOTEBOOK1_COLUMNS:
        errors.append(
            "Notebook 01 schema mismatch. "
            f"Expected {EXPECTED_NOTEBOOK1_COLUMNS}, "
            f"found {list(frame.columns)}."
        )

    for column in EXPECTED_NOTEBOOK1_COLUMNS:
        if column in frame.columns:
            if not pd.api.types.is_numeric_dtype(
                frame[column]
            ):
                errors.append(
                    f"{column} must be numeric."
                )

    # ------------------------------------------------------------
    # UTC index
    # ------------------------------------------------------------

    if not isinstance(frame.index, pd.DatetimeIndex):
        errors.append(
            "Notebook 01 output must use a DatetimeIndex."
        )

    else:
        if frame.index.tz is None:
            errors.append(
                "Notebook 01 index is timezone-naive."
            )

        elif str(frame.index.tz) != CANONICAL_TIMEZONE:
            errors.append(
                f"Expected UTC index, found {frame.index.tz}."
            )

        if not frame.index.is_monotonic_increasing:
            errors.append(
                "Notebook 01 index is not sorted."
            )

        duplicate_count = int(
            frame.index.duplicated().sum()
        )

        if duplicate_count:
            errors.append(
                f"Found {duplicate_count} duplicate timestamps."
            )

        differences = (
            frame.index
            .to_series()
            .diff()
            .dropna()
        )

        non_hourly = differences[
            differences
            != pd.Timedelta(hours=1)
        ]

        if not non_hourly.empty:
            errors.append(
                f"Found {len(non_hourly)} non-hourly gaps."
            )

    # ------------------------------------------------------------
    # Missing and infinite values
    # ------------------------------------------------------------

    for column in EXPECTED_NOTEBOOK1_COLUMNS:
        if column in frame.columns:
            non_missing = frame[column].dropna()

            if np.isinf(non_missing.to_numpy()).any():
                errors.append(
                    f"{column} contains infinite values."
                )

    price_missing = int(
        frame["price_eur_mwh"].isna().sum()
    )

    actual_load_missing = int(
        frame["load_actual_mwh"].isna().sum()
    )

    forecast_load_missing = int(
        frame["load_forecast_mwh"].isna().sum()
    )

    if price_missing:
        errors.append(
            f"Price contains {price_missing} missing values."
        )

    if actual_load_missing:
        errors.append(
            "Actual load contains "
            f"{actual_load_missing} missing values."
        )

    if (
        expected_forecast_nans is not None
        and forecast_load_missing
        != expected_forecast_nans
    ):
        errors.append(
            "Forecast-load missing count changed: "
            f"expected {expected_forecast_nans}, "
            f"found {forecast_load_missing}."
        )

    if errors:
        raise ValueError(
            "Notebook 01 input validation failed:\n- "
            + "\n- ".join(errors)
        )

    return {
        "rows": int(len(frame)),
        "columns": int(len(frame.columns)),
        "start_utc": frame.index.min(),
        "end_utc": frame.index.max(),
        "timezone": str(frame.index.tz),
        "duplicate_timestamps": int(
            frame.index.duplicated().sum()
        ),
        "non_hourly_gaps": 0,
        "missing_price": price_missing,
        "missing_actual_load": actual_load_missing,
        "missing_forecast_load": forecast_load_missing,
    }


notebook1_validation = validate_notebook1_input(
    clean_data,
    expected_forecast_nans=(
        EXPECTED_HISTORICAL_FORECAST_NANS
    ),
)

display(clean_data.head())

display(
    pd.Series(
        notebook1_validation,
        name="Notebook 01 input validation",
    )
)

print(
    "Minimum clean history required for live inference:",
    MINIMUM_CLEAN_HISTORY_HOURS,
    "hours",
)

,price_eur_mwh,load_actual_mwh,load_forecast_mwh
timestamp,,,
2022-08-31 19:00:00+00:00,677.26,54065.25,54673.50
2022-08-31 20:00:00+00:00,563.52,50545.00,50893.75
2022-08-31 21:00:00+00:00,532.51,46610.50,47281.75
2022-08-31 22:00:00+00:00,490.77,43779.50,45334.25
2022-08-31 23:00:00+00:00,484.36,42271.75,43953.25


rows                                         17568
columns                                          3
start_utc                2022-08-31 19:00:00+00:00
end_utc                  2024-09-01 18:00:00+00:00
timezone                                       UTC
duplicate_timestamps                             0
non_hourly_gaps                                  0
missing_price                                    0
missing_actual_load                              0
missing_forecast_load                           97
Name: Notebook 01 input validation, dtype: object

Minimum clean history required for live inference: 336 hours


In [3]:
ENGINEERED_FEATURE_COLUMNS = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "weekend",
    "price_lag_1",
    "price_lag_24",
    "price_lag_48",
    "price_lag_168",
    "price_roll_mean_24",
    "price_roll_std_24",
    "price_roll_mean_168",
    "price_roll_std_168",
]

# This preserves the old variable name for downstream compatibility.
feature_cols = ENGINEERED_FEATURE_COLUMNS.copy()


def build_features(clean_frame):
    """
    Create deterministic, leakage-safe features from the canonical
    Notebook 01 UTC dataset.

    The function does not drop rows and does not impute source
    missing values. Instead, it creates explicit validity flags.
    """
    validate_notebook1_input(
        clean_frame,
        expected_forecast_nans=None,
    )

    result = clean_frame.copy()

    # UTC remains the canonical index. German local time is used only
    # to calculate electricity-market calendar features.
    local_time = result.index.tz_convert(
        CALENDAR_TIMEZONE
    )

    # ------------------------------------------------------------
    # German local-time calendar features
    # ------------------------------------------------------------

    result["hour_sin"] = np.sin(
        2 * np.pi * local_time.hour / 24
    )

    result["hour_cos"] = np.cos(
        2 * np.pi * local_time.hour / 24
    )

    result["dow_sin"] = np.sin(
        2 * np.pi * local_time.dayofweek / 7
    )

    result["dow_cos"] = np.cos(
        2 * np.pi * local_time.dayofweek / 7
    )

    result["weekend"] = (
        local_time.dayofweek >= 5
    ).astype("int8")

    # ------------------------------------------------------------
    # Historical-price lag features
    # ------------------------------------------------------------

    for lag in PRICE_LAGS:
        result[f"price_lag_{lag}"] = (
            result["price_eur_mwh"].shift(lag)
        )

    # ------------------------------------------------------------
    # Past-only rolling-price features
    # ------------------------------------------------------------

    # shift(1) guarantees that the current row's price is excluded
    # from every rolling statistic.
    past_price = result["price_eur_mwh"].shift(1)

    for window in ROLLING_WINDOWS:
        result[f"price_roll_mean_{window}"] = (
            past_price
            .rolling(
                window=window,
                min_periods=window,
            )
            .mean()
        )

        result[f"price_roll_std_{window}"] = (
            past_price
            .rolling(
                window=window,
                min_periods=window,
            )
            .std(ddof=1)
        )

    # ------------------------------------------------------------
    # Validity flags
    # ------------------------------------------------------------

    # Preserves the meaning of the original features_valid column:
    # all engineered calendar/lag/rolling features are present.
    result["features_valid"] = (
        result[ENGINEERED_FEATURE_COLUMNS]
        .notna()
        .all(axis=1)
    )

    # Retained for compatibility and diagnostics.
    result["forecast_load_available"] = (
        result["load_forecast_mwh"].notna()
    )

    result["actual_load_available"] = (
        result["load_actual_mwh"].notna()
    )

    # This is the safer flag for later window generation.
    # A row is model-ready only when the engineered features and both
    # historical load variables are available.
    result["model_row_valid"] = (
        result["features_valid"]
        & result["forecast_load_available"]
        & result["actual_load_available"]
        & result["price_eur_mwh"].notna()
    )

    return result


data = build_features(clean_data)

display(data.head())

print(
    "Feature table shape:",
    data.shape,
)

print(
    "Engineered feature columns:",
    len(ENGINEERED_FEATURE_COLUMNS),
)

,price_eur_mwh,load_actual_mwh,load_forecast_mwh,hour_sin,hour_cos,dow_sin,dow_cos,weekend,price_lag_1,price_lag_24,price_lag_48,price_lag_168,price_roll_mean_24,price_roll_std_24,price_roll_mean_168,price_roll_std_168,features_valid,forecast_load_available,actual_load_available,model_row_valid
timestamp,,,,,,,,,,,,,,,,,,,,
2022-08-31 19:00:00+00:00,677.26,54065.25,54673.50,-0.707107,0.707107,0.974928,-0.222521,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,False
2022-08-31 20:00:00+00:00,563.52,50545.00,50893.75,-0.500000,0.866025,0.974928,-0.222521,0,677.26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,False
2022-08-31 21:00:00+00:00,532.51,46610.50,47281.75,-0.258819,0.965926,0.974928,-0.222521,0,563.52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,False
2022-08-31 22:00:00+00:00,490.77,43779.50,45334.25,0.000000,1.000000,0.433884,-0.900969,0,532.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,False
2022-08-31 23:00:00+00:00,484.36,42271.75,43953.25,0.258819,0.965926,0.433884,-0.900969,0,490.77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,True,False


Feature table shape: (17568, 20)
Engineered feature columns: 13


In [4]:
def validate_feature_dataset(
    clean_frame,
    feature_frame,
):
    errors = []

    # ------------------------------------------------------------
    # Notebook 01 data must remain unchanged
    # ------------------------------------------------------------

    if not feature_frame.index.equals(
        clean_frame.index
    ):
        errors.append(
            "Feature engineering changed the UTC index."
        )

    for column in EXPECTED_NOTEBOOK1_COLUMNS:
        try:
            pd.testing.assert_series_equal(
                feature_frame[column],
                clean_frame[column],
                check_names=True,
                check_dtype=True,
            )
        except AssertionError as error:
            errors.append(
                f"Base column {column} was modified: {error}"
            )

    # ------------------------------------------------------------
    # Feature-column existence and dtypes
    # ------------------------------------------------------------

    missing_features = [
        column
        for column in ENGINEERED_FEATURE_COLUMNS
        if column not in feature_frame.columns
    ]

    if missing_features:
        errors.append(
            f"Missing engineered features: {missing_features}"
        )

    for column in ENGINEERED_FEATURE_COLUMNS:
        if column not in feature_frame.columns:
            continue

        if not pd.api.types.is_numeric_dtype(
            feature_frame[column]
        ):
            errors.append(
                f"{column} is not numeric."
            )

        non_missing = feature_frame[column].dropna()

        if np.isinf(non_missing.to_numpy()).any():
            errors.append(
                f"{column} contains infinite values."
            )

    # ------------------------------------------------------------
    # Exact lag validation
    # ------------------------------------------------------------

    for lag in PRICE_LAGS:
        expected_lag = (
            clean_frame["price_eur_mwh"]
            .shift(lag)
        )

        actual_lag = feature_frame[
            f"price_lag_{lag}"
        ]

        try:
            pd.testing.assert_series_equal(
                actual_lag,
                expected_lag,
                check_names=False,
                check_dtype=True,
            )
        except AssertionError as error:
            errors.append(
                f"price_lag_{lag} is misaligned: {error}"
            )

    # ------------------------------------------------------------
    # Calendar-cycle validation
    # ------------------------------------------------------------

    hour_circle = (
        feature_frame["hour_sin"] ** 2
        + feature_frame["hour_cos"] ** 2
    )

    dow_circle = (
        feature_frame["dow_sin"] ** 2
        + feature_frame["dow_cos"] ** 2
    )

    if not np.allclose(
        hour_circle,
        1.0,
        atol=1e-10,
    ):
        errors.append(
            "Hour sine/cosine values do not lie on a unit circle."
        )

    if not np.allclose(
        dow_circle,
        1.0,
        atol=1e-10,
    ):
        errors.append(
            "Weekday sine/cosine values do not lie on a unit circle."
        )

    if not set(
        feature_frame["weekend"].unique()
    ).issubset({0, 1}):
        errors.append(
            "Weekend must contain only 0 and 1."
        )

    # ------------------------------------------------------------
    # Validity-flag validation
    # ------------------------------------------------------------

    expected_features_valid = (
        feature_frame[ENGINEERED_FEATURE_COLUMNS]
        .notna()
        .all(axis=1)
    )

    if not feature_frame[
        "features_valid"
    ].equals(expected_features_valid):
        errors.append(
            "features_valid does not match the engineered columns."
        )

    expected_model_row_valid = (
        expected_features_valid
        & feature_frame["price_eur_mwh"].notna()
        & feature_frame["load_actual_mwh"].notna()
        & feature_frame["load_forecast_mwh"].notna()
    )

    if not feature_frame[
        "model_row_valid"
    ].equals(expected_model_row_valid):
        errors.append(
            "model_row_valid is incorrectly calculated."
        )

    if errors:
        raise ValueError(
            "Feature validation failed:\n- "
            + "\n- ".join(errors)
        )

    return {
        "total_rows": int(len(feature_frame)),
        "engineered_feature_count": int(
            len(ENGINEERED_FEATURE_COLUMNS)
        ),
        "feature_warmup_rows": int(
            (~feature_frame["features_valid"]).sum()
        ),
        "forecast_load_missing_rows": int(
            (~feature_frame[
                "forecast_load_available"
            ]).sum()
        ),
        "model_valid_rows": int(
            feature_frame["model_row_valid"].sum()
        ),
        "model_invalid_rows": int(
            (~feature_frame["model_row_valid"]).sum()
        ),
        "minimum_clean_history_hours": int(
            MINIMUM_CLEAN_HISTORY_HOURS
        ),
    }


feature_validation_report = validate_feature_dataset(
    clean_frame=clean_data,
    feature_frame=data,
)

display(
    pd.Series(
        feature_validation_report,
        name="Notebook 02 validation",
    )
)

total_rows                     17568
engineered_feature_count          13
feature_warmup_rows              168
forecast_load_missing_rows        97
model_valid_rows               17303
model_invalid_rows               265
minimum_clean_history_hours      336
Name: Notebook 02 validation, dtype: int64

In [5]:
def build_operational_feature_history(
    clean_frame,
    forecast_origin_utc,
    lookback_hours=LOOKBACK_HOURS,
):
    """
    Produce the final engineered history preceding one forecast
    origin.

    A 168-row model window requires 336 clean raw hours because the
    earliest model-input row needs a 168-hour lag/rolling history.
    """
    origin = pd.Timestamp(forecast_origin_utc)

    if origin.tzinfo is None:
        origin = origin.tz_localize("UTC")
    else:
        origin = origin.tz_convert("UTC")

    if origin != origin.floor("h"):
        raise ValueError(
            "Forecast origin must be aligned to a full hour."
        )

    clean_history_start = origin - pd.Timedelta(
        hours=(
            lookback_hours
            + MAX_FEATURE_HISTORY_HOURS
        )
    )

    expected_clean_index = pd.date_range(
        start=clean_history_start,
        periods=(
            lookback_hours
            + MAX_FEATURE_HISTORY_HOURS
        ),
        freq="h",
    )

    missing_clean_timestamps = (
        expected_clean_index.difference(
            clean_frame.index
        )
    )

    if len(missing_clean_timestamps) > 0:
        raise ValueError(
            "Missing timestamps in operational clean history: "
            f"{missing_clean_timestamps[:10].tolist()}"
        )

    clean_history = clean_frame.loc[
        expected_clean_index
    ].copy()

    engineered_history = build_features(
        clean_history
    )

    model_input_start = origin - pd.Timedelta(
        hours=lookback_hours
    )

    expected_model_index = pd.date_range(
        start=model_input_start,
        periods=lookback_hours,
        freq="h",
    )

    model_history = engineered_history.loc[
        expected_model_index
    ].copy()

    if len(model_history) != lookback_hours:
        raise ValueError(
            f"Expected {lookback_hours} engineered rows, "
            f"received {len(model_history)}."
        )

    if not model_history[
        "model_row_valid"
    ].all():
        invalid_rows = model_history.loc[
            ~model_history["model_row_valid"],
            [
                "features_valid",
                "actual_load_available",
                "forecast_load_available",
            ],
        ]

        raise ValueError(
            "The operational window contains invalid model rows:\n"
            f"{invalid_rows.head(20).to_string()}"
        )

    return clean_history, model_history

In [6]:
# Use the latest historical 12:00 UTC origin that allows a complete
# 336-hour clean-history test.

candidate_origin = (
    clean_data.index.max().normalize()
    + pd.Timedelta(
        hours=FORECAST_ORIGIN_HOUR_UTC
    )
)

if (
    candidate_origin - pd.Timedelta(hours=1)
    > clean_data.index.max()
):
    candidate_origin -= pd.Timedelta(days=1)


demo_clean_history = None
demo_model_history = None
demo_origin = None

for days_back in range(60):
    current_origin = (
        candidate_origin
        - pd.Timedelta(days=days_back)
    )

    try:
        (
            demo_clean_history,
            demo_model_history,
        ) = build_operational_feature_history(
            clean_frame=clean_data,
            forecast_origin_utc=current_origin,
        )

        demo_origin = current_origin
        break

    except ValueError:
        continue


if demo_model_history is None:
    raise ValueError(
        "No complete historical 12:00 UTC operational window "
        "was found within the last 60 days."
    )


print("Demo forecast origin UTC:", demo_origin)

print(
    "Demo forecast origin Germany:",
    demo_origin.tz_convert(CALENDAR_TIMEZONE),
)

print(
    "Clean history rows used:",
    len(demo_clean_history),
)

print(
    "Final engineered model rows:",
    len(demo_model_history),
)

print(
    "Model input start:",
    demo_model_history.index.min(),
)

print(
    "Model input end:",
    demo_model_history.index.max(),
)

Demo forecast origin UTC: 2024-09-01 12:00:00+00:00
Demo forecast origin Germany: 2024-09-01 14:00:00+02:00
Clean history rows used: 336
Final engineered model rows: 168
Model input start: 2024-08-25 12:00:00+00:00
Model input end: 2024-09-01 11:00:00+00:00


In [7]:
feature_engineering_config = {
    "schema_version": "1.0",
    "input_file": INPUT_PATH.name,
    "output_file": FEATURE_OUTPUT_PATH.name,
    "canonical_timezone": CANONICAL_TIMEZONE,
    "calendar_timezone": CALENDAR_TIMEZONE,
    "scheduled_forecast_origin_hour_utc": (
        FORECAST_ORIGIN_HOUR_UTC
    ),
    "lookback_hours": LOOKBACK_HOURS,
    "forecast_horizon_hours": (
        FORECAST_HORIZON_HOURS
    ),
    "price_lags": PRICE_LAGS,
    "rolling_windows": ROLLING_WINDOWS,
    "rolling_target_shift_hours": 1,
    "rolling_std_ddof": 1,
    "maximum_feature_history_hours": (
        MAX_FEATURE_HISTORY_HOURS
    ),
    "minimum_clean_history_hours": (
        MINIMUM_CLEAN_HISTORY_HOURS
    ),
    "notebook1_columns": (
        EXPECTED_NOTEBOOK1_COLUMNS
    ),
    "engineered_feature_columns": (
        ENGINEERED_FEATURE_COLUMNS
    ),
    "validity_columns": [
        "features_valid",
        "forecast_load_available",
        "actual_load_available",
        "model_row_valid",
    ],
    "missing_value_policy": {
        "price": "must_be_complete",
        "actual_load": "must_be_complete_for_model_input",
        "forecast_load": (
            "preserve_nan_and_reject_affected_model_windows"
        ),
        "engineered_features": (
            "preserve_warmup_nan_and_use_validity_flags"
        ),
    },
}

# Save the full historical feature dataset.
data.to_parquet(
    FEATURE_OUTPUT_PATH,
    index=True,
)

# Save the exact feature-engineering definition.
with FEATURE_CONFIG_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_engineering_config,
        file,
        indent=2,
    )

# Reload and verify that serialization preserved the dataset.
reloaded_features = pd.read_parquet(
    FEATURE_OUTPUT_PATH
)

pd.testing.assert_frame_equal(
    data,
    reloaded_features,
    check_dtype=True,
    check_names=True,
)

validate_feature_dataset(
    clean_frame=clean_data,
    feature_frame=reloaded_features,
)

print("Saved and verified:")
print(FEATURE_OUTPUT_PATH)

print("\nSaved feature configuration:")
print(FEATURE_CONFIG_PATH)

print("\nFeature table shape:")
print(reloaded_features.shape)

Saved and verified:
/kaggle/working/smard_features.parquet

Saved feature configuration:
/kaggle/working/feature_engineering_config.json

Feature table shape:
(17568, 20)
